Fraud Detection - Shipment/Return Round-Trip Analysis
Identifies month-end shipments reversed as returns shortly after, using exact lot matching. Flags candidates for human review, not automated accusation.

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.fraud_detection import match_shipment_return_pairs, summarize_by_salesperson, summarize_by_customer
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load analysis silver

In [0]:
analysis_silver = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver["posting_date"] = pd.to_datetime(analysis_silver["posting_date"])
print(f"Analysis silver: {analysis_silver.shape}")

Find exact-lot matches

In [0]:
lot_matches = match_shipment_return_pairs(analysis_silver)
print(f"Matched shipment/return pairs: {len(lot_matches)}")

Customer-level summary

In [0]:
customer_fraud_summary = summarize_by_customer(lot_matches, analysis_silver)
print(f"Customers analyzed: {len(customer_fraud_summary)}")

Apply the minimum-sample-size filter and flag candidates automatically

In [0]:
flagged_candidates = customer_fraud_summary[
    (customer_fraud_summary["total_month_end_shipments"] >= fraud_min_shipments) &
    (customer_fraud_summary["round_trip_rate"] >= fraud_rate_threshold)
].sort_values("round_trip_rate", ascending=False)

Save

In [0]:
save_gold(blob_service, customer_fraud_summary, f"{ANALYSIS_BASE}/fraud_customer_summary.parquet")
save_gold(blob_service, lot_matches, f"{ANALYSIS_BASE}/fraud_flagged_pairs.parquet")

# Evidence specifically for flagged candidates
flagged_customer_names = flagged_candidates["customer_name"].tolist()
flagged_evidence = lot_matches[lot_matches["customer_name_shipment"].isin(flagged_customer_names)]

buffer = io.BytesIO()
flagged_evidence.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_evidence.xlsx")
blob_client.upload_blob(buffer, overwrite=True)

buffer2 = io.BytesIO()
customer_fraud_summary.to_excel(buffer2, index=False, engine="openpyxl")
buffer2.seek(0)
blob_client2 = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_customer_summary.xlsx")
blob_client2.upload_blob(buffer2, overwrite=True)

print(f"Saved evidence for {len(flagged_customer_names)} flagged accounts")

An alert when NEW flagged candidates appear

In [0]:
try:
    blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_customers_previous.json")
    previous_flagged = set(pd.read_json(io.BytesIO(blob_client.download_blob().readall()))["customer_name"])
except Exception:
    previous_flagged = set()

new_flags = set(flagged_candidates["customer_name"]) - previous_flagged
if new_flags:
    print(f"NEW flagged accounts since last run: {new_flags}")

blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/fraud_flagged_customers_previous.json")
blob_client.upload_blob(flagged_candidates[["customer_name"]].to_json(orient="records"), overwrite=True)